# Đề tài 2: Phân Đoạn Ảnh Y Tế (Kvasir-SEG) — Thiết Lập Trên Kaggle


In [ ]:
# 1. Kiểm tra môi trường Kaggle và phần cứng GPU (T4 x2 / P100)
!nvidia-smi

import os
import torch

print("\n--- Thông tin Môi trường & GPU ---")
print(f"Kaggle Runtime   : {os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Local / Other')}")
print(f"Working Dir      : {os.getcwd()}")
print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"GPU Count        : {num_gpus}")
    for idx in range(num_gpus):
        vram_gb = torch.cuda.get_device_properties(idx).total_memory / (1024 ** 3)
        print(f"  [GPU {idx}] {torch.cuda.get_device_name(idx)} ({vram_gb:.2f} GB VRAM)")
else:
    print("Error")


## 2. Clone / Đồng Bộ Repository Nhóm Vào `/kaggle/working`
Tải mã nguồn dự án từ [GitHub DL-Group7](https://github.com/daoviet2/DL-Group7.git) vào thư mục ghi được `/kaggle/working/DL-Group7`.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/daoviet2/DL-Group7.git"
WORK_BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
REPO_DIR = os.path.join(WORK_BASE, "DL-Group7")

if os.path.exists("configs/base.yaml"):
    print("[Git] Đang ở sẵn trong thư mục gốc dự án:", os.getcwd())
    subprocess.run(["git", "pull", "origin", "main"], check=True)
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    print("[Git] Đã chuyển vào thư mục dự án:", os.getcwd())
    subprocess.run(["git", "pull", "origin", "main"], check=True)
else:
    os.chdir(WORK_BASE)
    print(f"[Git] Đang clone repository từ {REPO_URL} vào {WORK_BASE} ...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    print("[Git] Thư mục làm việc hiện tại:", os.getcwd())

subprocess.run(["git", "status", "-s"], check=True)


## 3. Cài Đặt Thư Viện Phụ Thuộc Cho Cả 3 Nhánh (CNN, SegFormer, SAM)
Trên Kaggle đã cài sẵn PyTorch + CUDA, NumPy, Pandas và OpenCV; lệnh dưới chỉ cài bổ sung các gói còn thiếu trong `requirements.txt` (`segmentation-models-pytorch`, `evaluate`, v.v.).


## 3.1. Kết Nối & Đăng Nhập Weights & Biases (WandB)
Đăng nhập WandB từ Kaggle Secrets để đồng bộ thời gian thực đường cong huấn luyện và xuất link dashboard công khai nộp giảng viên.


In [ ]:
# Đăng nhập WandB từ Kaggle Secrets (hoặc nhập API key thủ công)
import wandb

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

In [ ]:
!pip install -r requirements.txt -q

import cv2
import albumentations as A
import segmentation_models_pytorch as smp
import transformers
import pandas as pd
import numpy as np

print("--- Xác nhận phiên bản thư viện trên Kaggle ---")
print("Albumentations :", A.__version__)
print("OpenCV         :", cv2.__version__)
print("SMP (U-Net)    :", smp.__version__)
print("Transformers   :", transformers.__version__)
print("[OK] Môi trường thư viện đã sẵn sàng!")


## 4. Kết Nối Dữ Liệu Kvasir-SEG (`/kaggle/input` hoặc Tự Động Tải)
Hệ thống tự động quét `/kaggle/input` nếu bạn đã gắn dataset trên Kaggle; nếu chưa có sẽ tự động tải và giải nén vào `./Kvasir-SEG`.


In [ ]:
import os
import sys

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from scripts.download_data import download_and_extract
from src.datasets.kvasir import resolve_kvasir_root

data_dir = download_and_extract()
resolved_dir = resolve_kvasir_root(data_dir)

img_count = len([f for f in os.listdir(os.path.join(resolved_dir, "images")) if f.lower().endswith((".jpg", ".png"))])
mask_count = len([f for f in os.listdir(os.path.join(resolved_dir, "masks")) if f.lower().endswith((".jpg", ".png"))])

print(f"\n[Dữ liệu] Đường dẫn đang dùng : {resolved_dir}")
print(f"[Dữ liệu] Images: {img_count}/1000 | Masks: {mask_count}/1000")
assert img_count == 1000 and mask_count == 1000, "Dữ liệu chưa đủ 1000 cặp ảnh-mask!"
print("[OK] Bộ dữ liệu Kvasir-SEG toàn vẹn 100%.")


## 5. Xác Nhận Bộ Chia Tập Phân Tầng Cố Định (Stratified 80/10/10)
Mọi mô hình của cả 3 thành viên bắt buộc dùng chung `splits/train.txt` (800), `splits/val.txt` (100) và `splits/test.txt` (100).


In [ ]:
if not os.path.exists("splits/metadata.csv"):
    !python scripts/prepare_splits.py
else:
    df_meta = pd.read_csv("splits/metadata.csv")
    print(f"[Splits] Đã nạp splits/metadata.csv ({len(df_meta)} mẫu):")
    print(df_meta.groupby(["split", "size_category"]).size().unstack(fill_value=0))


## 6. Chạy Kiểm Thử Tự Động Pipeline Dữ Liệu (Sanity Check)
Xác thực `KvasirSegDataset`, kích thước tensor `[B, 3, 256, 256]`, tính nhị phân `{0.0, 1.0}` của mask và tọa độ `bbox` cho SAM.


In [ ]:
!python scripts/sanity_check_data.py


## 7. Trực Quan Hóa Mini-Batch Trên Kaggle Notebook
Hiển thị trực tiếp 4 mẫu từ `train_loader` gồm: **Ảnh gốc kèm Ground Truth BBox | Ground Truth Mask | Overlay vùng Polyp**.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from src.datasets.kvasir import get_dataloaders
from src.utils.config import load_config
from src.utils.vis import create_overlay, draw_bounding_box
from src.datasets.transforms import denormalize_image

cfg = load_config("configs/base.yaml")
cfg_demo = cfg.copy()
cfg_demo["data"] = cfg["data"].copy()
cfg_demo["data"]["num_workers"] = 2
cfg_demo["data"]["batch_size"] = 4

train_loader, val_loader, test_loader = get_dataloaders(cfg_demo)
batch = next(iter(train_loader))

images = batch["image"]
masks = batch["mask"]
sample_ids = batch["id"]
bboxes = batch["bbox"]

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for i in range(4):
    img_denorm = denormalize_image(images[i])
    mask_np = masks[i, 0].numpy()

    img_box = draw_bounding_box(img_denorm, bboxes[i], color=(255, 50, 50), thickness=2)
    overlay = create_overlay(img_denorm, mask_np, color=(0, 230, 80), alpha=0.45)

    axes[i, 0].imshow(img_box)
    axes[i, 0].set_title(f"Ảnh gốc (BBox): {sample_ids[i]}", fontsize=11)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask_np, cmap="gray")
    axes[i, 1].set_title("Ground Truth Mask", fontsize=11)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("Overlay Phân Đoạn", fontsize=11)
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()


## 8. Kiểm Thử Forward Pass Trên GPU Kaggle & Thư Mục Xuất Kết Quả
Mọi file log `.csv`, hình ảnh và checkpoint `.pth` lưu trong `/kaggle/working/DL-Group7/outputs` sẽ xuất hiện tại mục **Output** bên phải của Kaggle để tải về hoặc lưu qua *Save Version*.


In [ ]:
if torch.cuda.is_available():
    import time
    device = torch.device("cuda")

    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
    ).to(device)

    real_images = batch["image"].to(device)
    torch.cuda.synchronize()
    start_time = time.time()
    with torch.no_grad():
        logits = model(real_images)
    torch.cuda.synchronize()
    elapsed = (time.time() - start_time) * 1000

    allocated_mem = torch.cuda.memory_allocated(device) / (1024 ** 2)
    print("--- Kiểm Thử Forward Pass GPU Kaggle Hoàn Tất ---")
    print(f"Batch đầu vào        : {real_images.shape}")
    print(f"Logits đầu ra        : {logits.shape}")
    print(f"Thời gian Forward    : {elapsed:.2f} ms")
    print(f"VRAM cấp phát        : {allocated_mem:.2f} MB")
    print(f"Thư mục lưu Outputs  : {os.path.abspath('outputs')}")
    print("[OK] Môi trường Kaggle GPU đã sẵn sàng 100% cho cả 3 thành viên!")
else:
    print("Đang chạy trên CPU. Hãy bật GPU T4 x2 / P100 trong Kaggle Session options khi huấn luyện!")


## 8.5. Overfit Sanity Check Trên 10 Ảnh Cố Định
Trước khi huấn luyện 50 epoch, chạy sanity check trên 10 ảnh cố định với mô hình U-Net từ đầu và hàm mất mát `BCE + SoftDice`.
Mục tiêu: Đạt Dice >= 0.95 ở chế độ `model.eval()`, xuất hình trực quan hóa 4 cột so sánh GT vs. Dự đoán vào `outputs/overfit/overfit_10.png`.


In [ ]:
# Chạy kiểm tra overfit trên 10 ảnh cố định (~15-25 giây trên GPU T4)
!python scripts/overfit_sanity.py --num-images 10 --iters 400


## 9. Huấn Luyện Baseline U-Net Cài Đặt Từ Đầu (Scratch) + Đồng Bộ WandB
Chạy huấn luyện mô hình U-Net từ đầu với hàm mất mát `BCE + SoftDice` trong 50 epoch (seed 42, ảnh 256×256, bật AMP fp16).
Đường cong huấn luyện (Loss, Dice, IoU, GPU Memory) sẽ được đồng bộ trực tiếp lên dashboard công khai của **Weights & Biases (WandB)** theo thời gian thực.


In [ ]:
# Huấn luyện Baseline U-Net từ đầu và đẩy log trực tiếp lên WandB
!python scripts/train.py --config configs/unet_scratch_bce_dice.yaml --override logging.wandb_mode=online


## 10. Đánh Giá Phân Tầng Theo Kích Thước Polyp (Small, Medium, Large)
Nạp checkpoint tốt nhất (`best.pth`), đánh giá trên tập Validation, xuất file điểm chi tiết từng ảnh (`outputs/predictions/..._per_image.csv`) và in bảng phân tích hiệu năng phục vụ viết báo cáo.


In [ ]:
# Đánh giá checkpoint tốt nhất và phân tích hiệu năng theo 3 nhóm kích thước polyp
!python scripts/evaluate.py --checkpoint outputs/checkpoints/unet_scratch_bce_dice_s42_256/best.pth --split val


## 11. Bảng Tổng Hợp Kết Quả & Link WandB Dashboard
Đọc và hiển thị nội dung file `outputs/results.csv` chứa thông số mô hình, điểm Dice/IoU, thời gian huấn luyện và link WandB chính thức nộp giảng viên.


In [ ]:
import os
import pandas as pd

results_path = "outputs/results.csv"
if os.path.exists(results_path):
    df_results = pd.read_csv(results_path)
    display(df_results)
else:
    print(f"Chưa tìm thấy file {results_path}. Hãy chạy cell huấn luyện ở trên trước!")


## 12. Trực Quan Hóa Đường Cong Huấn Luyện (Loss, Dice, IoU & Learning Rate)
Đọc file nhật ký epoch (`outputs/logs/unet_scratch_bce_dice_s42_256.csv`) và vẽ biểu đồ đối sánh Train vs. Val để theo dõi quá trình hội tụ và hiện tượng quá khớp (overfitting).


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

log_path = "outputs/logs/unet_scratch_bce_dice_s42_256.csv"
if os.path.exists(log_path):
    df_log = pd.read_csv(log_path)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Loss Curves
    axes[0].plot(df_log["epoch"], df_log["train_loss"], label="Train Loss", color="#1f77b4", linewidth=2)
    axes[0].plot(df_log["epoch"], df_log["val_loss"], label="Val Loss", color="#d62728", linewidth=2, linestyle="--")
    axes[0].set_title("Loss Curves (BCE + SoftDice)", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Epoch", fontsize=11)
    axes[0].set_ylabel("Loss", fontsize=11)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # 2. Dice & IoU Curves
    axes[1].plot(df_log["epoch"], df_log["train_dice"], label="Train Dice", color="#2ca02c", linewidth=2)
    axes[1].plot(df_log["epoch"], df_log["val_dice"], label="Val Dice", color="#ff7f0e", linewidth=2, linestyle="--")
    axes[1].plot(df_log["epoch"], df_log["val_iou"], label="Val IoU", color="#9467bd", linewidth=1.5, linestyle=":")
    axes[1].set_title("Macro Dice & IoU Curves", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Epoch", fontsize=11)
    axes[1].set_ylabel("Score", fontsize=11)
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    # 3. Learning Rate Schedule
    axes[2].plot(df_log["epoch"], df_log["lr"], label="Learning Rate", color="#8c564b", linewidth=2)
    axes[2].set_title("Learning Rate Schedule", fontsize=13, fontweight="bold")
    axes[2].set_xlabel("Epoch", fontsize=11)
    axes[2].set_ylabel("LR", fontsize=11)
    axes[2].ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
    axes[2].legend(fontsize=10)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    curve_img_path = "outputs/logs/unet_scratch_curves.png"
    plt.savefig(curve_img_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"[OK] Đã lưu đồ thị huấn luyện vào: {curve_img_path}")
else:
    print(f"Chưa tìm thấy file {log_path}. Hãy chạy huấn luyện trước!")


## 13. Trực Quan Hóa Mẫu Dự Đoán Trên Tập Validation Theo Phân Tầng Kích Thước
Trích xuất 6 mẫu tiêu biểu từ tập Validation (2 mẫu Small, 2 mẫu Medium, 2 mẫu Large) để kiểm tra chất lượng phân đoạn biên của U-Net từ đầu.


In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.models import build_model
from src.datasets.kvasir import build_dataloader
from src.datasets.transforms import denormalize_image
from src.utils.checkpoint import load_checkpoint
from src.utils.config import load_config

ckpt_path = "outputs/checkpoints/unet_scratch_bce_dice_s42_256/best.pth"
per_img_path = "outputs/predictions/unet_scratch_bce_dice_s42_256_val_per_image.csv"

if os.path.exists(ckpt_path) and os.path.exists(per_img_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg = load_config("configs/unet_scratch_bce_dice.yaml")
    
    model = build_model(cfg).to(device)
    load_checkpoint(ckpt_path, model=model, device=device)
    model.eval()
    
    df_pred = pd.read_csv(per_img_path)
    
    # Lấy đại diện 2 mẫu mỗi tầng: small, medium, large
    selected_ids = []
    for cat in ["small", "medium", "large"]:
        sub = df_pred[df_pred["size_category"] == cat]
        if len(sub) > 0:
            selected_ids.extend(sub["id"].head(2).tolist())
            
    val_loader = build_dataloader("val", cfg, shuffle=False, drop_last=False)
    
    sample_dict = {}
    with torch.no_grad():
        for batch in val_loader:
            for b_idx, s_id in enumerate(batch["id"]):
                if s_id in selected_ids and s_id not in sample_dict:
                    img_t = batch["image"][b_idx:b_idx+1].to(device)
                    mask_t = batch["mask"][b_idx:b_idx+1]
                    logit = model(img_t)
                    prob = torch.sigmoid(logit).squeeze().cpu().numpy()
                    pred_binary = (prob > 0.5).astype(np.uint8)
                    
                    rgb = denormalize_image(img_t.squeeze(0).cpu().numpy())
                    gt = mask_t.squeeze().cpu().numpy().astype(np.uint8)
                    
                    row_info = df_pred[df_pred["id"] == s_id].iloc[0]
                    sample_dict[s_id] = {
                        "rgb": rgb,
                        "gt": gt,
                        "pred": pred_binary,
                        "dice": row_info["dice"],
                        "iou": row_info["iou"],
                        "category": row_info["size_category"],
                    }
                    
    n_samples = len(sample_dict)
    fig, axes = plt.subplots(n_samples, 4, figsize=(16, 3.5 * n_samples))
    if n_samples == 1:
        axes = np.expand_dims(axes, 0)
        
    for i, (s_id, s_data) in enumerate(sample_dict.items()):
        rgb = s_data["rgb"]
        gt = s_data["gt"]
        pred = s_data["pred"]
        
        # Overlay: GT viền xanh lá (0, 255, 0), Pred viền đỏ (255, 0, 0)
        overlay = rgb.copy()
        gt_contours, _ = cv2.findContours(gt, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        pred_contours, _ = cv2.findContours(pred, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, gt_contours, -1, (0, 255, 0), 2)
        cv2.drawContours(overlay, pred_contours, -1, (255, 0, 0), 2)
        
        axes[i, 0].imshow(rgb)
        axes[i, 0].set_title(f"Ảnh nội soi [{s_data['category'].upper()}]\n{s_id}", fontsize=10)
        axes[i, 0].axis("off")
        
        axes[i, 1].imshow(gt, cmap="gray")
        axes[i, 1].set_title("Ground Truth Mask", fontsize=10)
        axes[i, 1].axis("off")
        
        axes[i, 2].imshow(pred, cmap="gray")
        axes[i, 2].set_title(f"Dự đoán (Threshold=0.5)\nDice: {s_data['dice']:.4f} | IoU: {s_data['iou']:.4f}", fontsize=10)
        axes[i, 2].axis("off")
        
        axes[i, 3].imshow(overlay)
        axes[i, 3].set_title("Overlay (Xanh: GT | Đỏ: Dự đoán)", fontsize=10)
        axes[i, 3].axis("off")
        
    plt.tight_layout()
    pred_fig_path = "outputs/predictions/unet_scratch_val_samples.png"
    plt.savefig(pred_fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"[OK] Đã lưu hình trực quan hóa vào: {pred_fig_path}")
else:
    print("Chưa có checkpoint hoặc per_image.csv.")


## 14. Tuần 3: Overfit Sanity Check U-Net ResNet34 Pretrained Trên 10 Ảnh Cố Định
Xác nhận pipeline forward/backward của U-Net ResNet34 Pretrained trên 10 ảnh cố định của `splits/train.txt`.
Mục tiêu: Đạt Dice eval >= 0.95, kiểm tra tốc độ hội tụ (số iter đạt Dice >= 0.99) nhanh hơn U-Net scratch (cần 325 iters) nhờ đặc trưng ImageNet sẵn có.
Kết quả xuất ra `outputs/overfit/overfit_10_resnet34.png` và `outputs/overfit/overfit_10_resnet34_log.csv`.


In [ ]:
# Chạy Overfit Sanity Check cho U-Net ResNet34 Pretrained (~15-20 giây trên GPU T4)
!python scripts/overfit_sanity.py --config configs/unet_resnet34_bce_dice.yaml --num-images 10 --iters 400 --lr 1e-3 --tag resnet34


## 15. Tuần 3: Huấn Luyện U-Net Scratch Thêm 2 Seed (Seed 100 & 2026
Chạy huấn luyện U-Net scratch cho 2 seed còn lại (seed 100 và seed 2026.
Tự động lưu checkpoint, đánh giá trên Validation và đồng bộ lên WandB.


In [ ]:
# 1. Huấn luyện U-Net Scratch - Seed 100 (~22-25 phút trên GPU T4)
!python scripts/train.py --config configs/unet_scratch_bce_dice.yaml --seed 100 --override logging.wandb_mode=online
!python scripts/evaluate.py --checkpoint outputs/checkpoints/unet_scratch_bce_dice_s100_256/best.pth --split val

# 2. Huấn luyện U-Net Scratch - Seed 2026 (~22-25 phút trên GPU T4)
!python scripts/train.py --config configs/unet_scratch_bce_dice.yaml --seed 2026 --override logging.wandb_mode=online
!python scripts/evaluate.py --checkpoint outputs/checkpoints/unet_scratch_bce_dice_s2026_256/best.pth --split val

# 3. Tự động tổng hợp kết quả 3 seed của U-Net Scratch (Mean ± Std & Phân tầng kích thước)
!python scripts/aggregate_results.py


## 16. Tuần 3: Huấn Luyện U-Net ResNet34 Pretrained Đầy Đủ 3 Seed (42, 100, 2026)
Huấn luyện U-Net với Pretrained Encoder ResNet34 (ImageNet) qua 3 seed cố định (42, 100, 2026).
Cấu hình: AdamW lr=3e-4, scheduler Cosine Warmup 3 epoch, early stopping patience=10, Mixed Precision AMP fp16.
Tự động đồng bộ đường cong huấn luyện và model artifact lên dashboard WandB `kvasir-seg-polyp`.


In [ ]:
# 1. Huấn luyện U-Net ResNet34 Pretrained - Seed 42
!python scripts/train.py --config configs/unet_resnet34_bce_dice.yaml --seed 42 --override logging.wandb_mode=online
!python scripts/evaluate.py --checkpoint outputs/checkpoints/unet_resnet34_bce_dice_s42_256/best.pth --split val

# 2. Huấn luyện U-Net ResNet34 Pretrained - Seed 100
!python scripts/train.py --config configs/unet_resnet34_bce_dice.yaml --seed 100 --override logging.wandb_mode=online
!python scripts/evaluate.py --checkpoint outputs/checkpoints/unet_resnet34_bce_dice_s100_256/best.pth --split val

# 3. Huấn luyện U-Net ResNet34 Pretrained - Seed 2026
!python scripts/train.py --config configs/unet_resnet34_bce_dice.yaml --seed 2026 --override logging.wandb_mode=online
!python scripts/evaluate.py --checkpoint outputs/checkpoints/unet_resnet34_bce_dice_s2026_256/best.pth --split val


## 17. Tuần 3: Tổng Hợp Bảng So Sánh Định Lượng: U-Net Scratch vs. U-Net ResNet34 Pretrained
Cell dưới đây đọc trực tiếp `outputs/results.csv` và các file `per_image.csv`, tự động tổng hợp:
1. **Bảng kết quả chính** (mean ± std qua 3 seed) cho cả 2 mô hình.
2. **Bảng phân tầng theo kích thước polyp**: Small (< 5%), Medium (5-20%), Large (>= 20%).
3. **Biểu đồ so sánh trực quan** hiệu năng và chi phí tính toán (số tham số, tốc độ suy luận).


In [ ]:
import glob
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

results_path = "outputs/results.csv"
if os.path.exists(results_path):
    df_res = pd.read_csv(results_path)
    val_df = df_res[df_res["split"] == "val"].copy()
    
    print("=" * 85)
    print("BẢNG 1: TỔNG HỢP SO SÁNH (MEAN ± STD QUA CÁC SEED TRÊN VALIDATION)")
    print("=" * 85)
    
    summary_rows = []
    for model_name, grp in val_df.groupby("model"):
        n_runs = len(grp)
        dice_m = grp["dice_mean"].mean()
        dice_s = grp["dice_mean"].std(ddof=1) if n_runs > 1 else 0.0
        iou_m = grp["iou_mean"].mean()
        iou_s = grp["iou_mean"].std(ddof=1) if n_runs > 1 else 0.0
        params = grp["n_params"].iloc[0]
        infer_spd = grp["infer_ms_per_img"].mean()
        fps = 1000.0 / infer_spd if infer_spd > 0 else 0.0
        train_t = grp["train_time_min"].mean()
        
        summary_rows.append({
            "Mô hình": model_name,
            "Số runs": n_runs,
            "Val Macro Dice": f"{dice_m:.4f} ± {dice_s:.4f}",
            "Val Macro IoU": f"{iou_m:.4f} ± {iou_s:.4f}",
            "Số tham số": f"{params:,}",
            "Tốc độ (ms/ảnh)": f"{infer_spd:.2f}",
            "FPS": f"{fps:.1f}",
            "TG Train (phút)": f"{train_t:.2f}",
        })
        
    df_summary = pd.DataFrame(summary_rows)
    display(df_summary)
    
    # 2. Phân tích phân tầng kích thước polyp từ per_image CSV
    print("\n" + "=" * 85)
    print("BẢNG 2: PHÂN TẦNG KÍCH THƯỚC POLYP (SMALL / MEDIUM / LARGE)")
    print("=" * 85)
    
    per_img_files = glob.glob("outputs/predictions/*_val_per_image.csv")
    all_per_img = []
    for pf in per_img_files:
        t_df = pd.read_csv(pf)
        model_tag = "unet_resnet34" if "resnet34" in pf else "unet_scratch"
        t_df["model_type"] = model_tag
        all_per_img.append(t_df)
        
    if all_per_img:
        df_all_imgs = pd.concat(all_per_img, ignore_index=True)
        strat_summary = []
        for m_tag, grp_m in df_all_imgs.groupby("model_type"):
            for cat in ["small", "medium", "large"]:
                sub_cat = grp_m[grp_m["size_category"] == cat]
                if len(sub_cat) > 0:
                    strat_summary.append({
                        "Mô hình": m_tag,
                        "Nhóm Kích thước": cat.upper(),
                        "Số mẫu đánh giá": len(sub_cat),
                        "Dice (Mean ± Std)": f"{sub_cat['dice'].mean():.4f} ± {sub_cat['dice'].std(ddof=1):.4f}",
                        "IoU (Mean ± Std)": f"{sub_cat['iou'].mean():.4f} ± {sub_cat['iou'].std(ddof=1):.4f}",
                    })
        df_strat = pd.DataFrame(strat_summary)
        display(df_strat)
        
        # 3. Biểu đồ trực quan so sánh
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        categories = ["small", "medium", "large"]
        scratch_data = [df_all_imgs[(df_all_imgs["model_type"] == "unet_scratch") & (df_all_imgs["size_category"] == c)]["dice"].values for c in categories]
        resnet_data = [df_all_imgs[(df_all_imgs["model_type"] == "unet_resnet34") & (df_all_imgs["size_category"] == c)]["dice"].values for c in categories]
        
        x = np.arange(len(categories))
        width = 0.35
        
        s_means = [np.mean(d) if len(d)>0 else 0 for d in scratch_data]
        r_means = [np.mean(d) if len(d)>0 else 0 for d in resnet_data]
        
        axes[0].bar(x - width/2, s_means, width, label="U-Net Scratch", color="#1f77b4", alpha=0.85)
        axes[0].bar(x + width/2, r_means, width, label="U-Net ResNet34", color="#2ca02c", alpha=0.85)
        axes[0].set_title("Macro Dice Theo Phân Tầng Kích Thước Polyp", fontsize=12, fontweight="bold")
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(["Small (<5%)", "Medium (5-20%)", "Large (>=20%)"], fontsize=10)
        axes[0].set_ylabel("Dice Trung Bình", fontsize=11)
        axes[0].set_ylim(0, 1.05)
        axes[0].legend(fontsize=10)
        axes[0].grid(True, axis="y", alpha=0.3)
        
        for i in range(len(x)):
            axes[0].text(x[i] - width/2, s_means[i] + 0.02, f"{s_means[i]:.3f}", ha="center", fontsize=9, fontweight="bold")
            axes[0].text(x[i] + width/2, r_means[i] + 0.02, f"{r_means[i]:.3f}", ha="center", fontsize=9, fontweight="bold")
            
        models = ["U-Net Scratch", "U-Net ResNet34"]
        params_m = [31.04, 24.44]
        axes[1].bar(models, params_m, color=["#1f77b4", "#ff7f0e"], width=0.45, alpha=0.85)
        axes[1].set_title("So Sánh Số Tham Số Mô Hình (Triệu Tham Số)", fontsize=12, fontweight="bold")
        axes[1].set_ylabel("Tham Số (Triệu - M)", fontsize=11)
        axes[1].set_ylim(0, 36)
        axes[1].grid(True, axis="y", alpha=0.3)
        for i, v in enumerate(params_m):
            axes[1].text(i, v + 0.8, f"{v:.2f}M", ha="center", fontsize=10, fontweight="bold")
            
        plt.tight_layout()
        comparison_plot = "outputs/predictions/scratch_vs_resnet34_stratified_comparison.png"
        plt.savefig(comparison_plot, dpi=200, bbox_inches="tight")
        plt.show()
        print(f"[OK] Đã lưu biểu đồ so sánh vào: {comparison_plot}")
else:
    print("Chưa có file results.csv. Hãy chạy huấn luyện các mô hình trước!")
